In [17]:
%pip install scikit-learn pandas numpy


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [18]:
import pandas as pd

TRAIN_DATA = pd.read_csv('train-data.csv', index_col='id')
TRAIN_LABEL = pd.read_csv('train-label.csv', index_col='id')
TEST_DATA = pd.read_csv('test-data.csv', index_col='id')

In [19]:
import numpy as np #16+3 col
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

if 'subscription' in TRAIN_DATA.columns:
    TRAIN_DATA = TRAIN_DATA.drop(columns=['subscription'])

def add_features(df):
    df = df.copy()
    # Was customer contacted recently? (within 30 days)
    df['contacted_recently'] = ((df['pdays'] != -1) & (df['pdays'] < 30)).astype(int)
    # Was previous campaign a success?
    df['prev_success'] = (df['poutcome'] == 'SUC').astype(int)
    # Long call = very engaged customer (> 5 mins)
    df['long_call'] = (df['duration'] > 300).astype(int)
    return df

TRAIN_DATA = add_features(TRAIN_DATA)
TEST_DATA = add_features(TEST_DATA)

cat_cols = ['job', 'marital_status', 'education', 'default_loan',
            'housing_loan', 'personal_loan', 'contact_type', 'poutcome']
num_cols = ['age', 'balance', 'day', 'month', 'duration',
            'campaign', 'pdays', 'previous',
            'contacted_recently', 'prev_success', 'long_call']  # new cols added here

def preprocess(df, encoder, scaler):
    df = df.copy()
    cat_enc = pd.DataFrame(
        encoder.transform(df[cat_cols]),
        columns=cat_cols,
        index=df.index
    )
    num_scaled = pd.DataFrame(
        scaler.transform(df[num_cols]),
        columns=num_cols,
        index=df.index
    )
    return pd.concat([cat_enc, num_scaled], axis=1)

ENCODER = OrdinalEncoder().fit(TRAIN_DATA[cat_cols])
SCALER = StandardScaler().fit(TRAIN_DATA[num_cols])

X_train = preprocess(TRAIN_DATA, ENCODER, SCALER)
X_test = preprocess(TEST_DATA, ENCODER, SCALER)
y_train = TRAIN_LABEL['subscription'].values

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)

X_train shape: (29839, 19)
X_test shape: (19893, 19)


In [20]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',  # handles class imbalance
    random_state=2328
)

cv_scores = cross_val_score(
    model, X_train, y_train,
    cv=skf,
    scoring='balanced_accuracy'
)
print(f'CV Balanced Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Per-fold scores: {cv_scores}')

CV Balanced Accuracy: 0.8056 ± 0.0050
Per-fold scores: [0.79882397 0.81393589 0.80254714 0.80599408 0.80694219]


In [21]:
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import balanced_accuracy_score

splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
I_train, I_val = next(splitter.split(X_train, y_train))
X_tr, X_val = X_train.iloc[I_train], X_train.iloc[I_val]
y_tr, y_val = y_train[I_train], y_train[I_val]

model.fit(X_tr, y_tr)
val_probs = model.predict_proba(X_val)[:, 1]

best_threshold, best_ba = 0.5, 0.0
for thresh in np.arange(0.1, 0.9, 0.01):
    preds = (val_probs >= thresh).astype(int)
    ba = balanced_accuracy_score(y_val, preds)
    if ba > best_ba:
        best_ba = ba
        best_threshold = thresh

print(f'Best threshold: {best_threshold:.2f}')
print(f'Best Val Balanced Accuracy: {best_ba:.4f}')

Best threshold: 0.45
Best Val Balanced Accuracy: 0.8223


In [22]:
final_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)

final_model.fit(X_train, y_train)

test_probs = final_model.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= best_threshold).astype(int)

print(f'Prediction distribution ~0: {(test_preds==0).sum()}, 1: {(test_preds==1).sum()}')

Prediction distribution ~0: 14459, 1: 5434


save to csv


In [27]:
submission = pd.DataFrame({
    'id': TEST_DATA.index,
    'subscription': test_preds
})

submission.to_csv('submission.csv', index=False)
print('Saved! Preview:')
print(submission.head())

Saved! Preview:
      id  subscription
0  37797             0
1  37798             0
2  37799             0
3  37800             0
4  37801             0


test with different column

In [24]:
import numpy as np #16+3 col
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

if 'subscription' in TRAIN_DATA.columns:
    TRAIN_DATA = TRAIN_DATA.drop(columns=['subscription'])

def add_features(df):
    df = df.copy()
    # Was customer contacted recently? (within 30 days)
    df['contacted_recently'] = ((df['pdays'] != -1) & (df['pdays'] < 30)).astype(int)
    # Was previous campaign a success?
    df['prev_success'] = (df['poutcome'] == 'SUC').astype(int)
    # Long call = very engaged customer (> 5 mins)
    df['long_call'] = (df['duration'] > 300).astype(int)
    return df

TRAIN_DATA = add_features(TRAIN_DATA)
TEST_DATA = add_features(TEST_DATA)

cat_cols = ['job', 'marital_status', 'education', 'default_loan',
            'housing_loan', 'personal_loan', 'contact_type', 'poutcome']
num_cols = ['age', 'balance', 'day', 'month', 'duration',
            'campaign', 'pdays', 'previous',
            'contacted_recently', 'prev_success', 'long_call']  # new cols added here

def preprocess(df, encoder, scaler):
    df = df.copy()
    cat_enc = pd.DataFrame(
        encoder.transform(df[cat_cols]),
        columns=cat_cols,
        index=df.index
    )
    num_scaled = pd.DataFrame(
        scaler.transform(df[num_cols]),
        columns=num_cols,
        index=df.index
    )
    return pd.concat([cat_enc, num_scaled], axis=1)

ENCODER = OrdinalEncoder().fit(TRAIN_DATA[cat_cols])
SCALER = StandardScaler().fit(TRAIN_DATA[num_cols])

X_train = preprocess(TRAIN_DATA, ENCODER, SCALER)
X_test = preprocess(TEST_DATA, ENCODER, SCALER)
y_train = TRAIN_LABEL['subscription'].values

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)

X_train shape: (29839, 19)
X_test shape: (19893, 19)


In [26]:
import numpy as np #16 col
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

# Drop subscription if it leaked into TRAIN_DATA
if 'subscription' in TRAIN_DATA.columns:
    TRAIN_DATA = TRAIN_DATA.drop(columns=['subscription'])
#consider all col 1
cat_cols = ['job', 'marital_status', 'education', 'default_loan',
            'housing_loan', 'personal_loan', 'contact_type', 'poutcome']
num_cols = ['age', 'balance', 'day', 'month', 'duration',
            'campaign', 'pdays', 'previous']

def preprocess(df, encoder, scaler):
    df = df.copy()
    # Encode categoricals
    cat_enc = pd.DataFrame(
        encoder.transform(df[cat_cols]),
        columns=cat_cols,
        index=df.index
    )
    # Scale numericals
    num_scaled = pd.DataFrame(
        scaler.transform(df[num_cols]),
        columns=num_cols,
        index=df.index
    )
    return pd.concat([cat_enc, num_scaled], axis=1)

# Fit encoder and scaler on training data only
ENCODER = OrdinalEncoder().fit(TRAIN_DATA[cat_cols])
SCALER = StandardScaler().fit(TRAIN_DATA[num_cols])

X_train = preprocess(TRAIN_DATA, ENCODER, SCALER)
X_test = preprocess(TEST_DATA, ENCODER, SCALER)
y_train = TRAIN_LABEL['subscription'].values

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)

X_train shape: (29839, 16)
X_test shape: (19893, 16)
